In [ ]:
import numpy as np
import matplotlib.pyplot as plt
from sklearn.datasets import load_wine
from sklearn.preprocessing import StandardScaler
from sklearn.metrics import completeness_score

In [ ]:
# 1. Prepare Data
data = load_wine()
X = data.data
y = data.target # We only use this at the VERY end for comparison

# Scaling is mandatory for K-Means because it's distance-based
scaler = StandardScaler()
X_scaled = scaler.fit_transform(X)

In [ ]:
# 2. K-Means Implementation from Scratch
class KMeansCustom:
    def __init__(self, K=3, max_iters=100, random_state=42):
        self.K = K
        self.max_iters = max_iters
        self.random_state = random_state
        self.centroids = None

    def fit(self, X):
        np.random.seed(self.random_state)
        # Initialization: Pick 3 random points as starting centers
        random_idx = np.random.choice(len(X), self.K, replace=False)
        self.centroids = X[random_idx]

        for i in range(self.max_iters):
            # A. Assignment: Find nearest centroid for each point
            # Formula: sqrt(sum((x - centroid)^2))
            distances = np.sqrt(((X[:, np.newaxis] - self.centroids)**2).sum(axis=2))
            labels = np.argmin(distances, axis=1)

            # B. Update: Move centroids to the mean of their assigned points
            new_centroids = np.array([X[labels == k].mean(axis=0) for k in range(self.K)])

            # C. Check for convergence (if centroids stop moving)
            if np.allclose(self.centroids, new_centroids):
                print(f"Converged at iteration {i}")
                break
            self.centroids = new_centroids
        
        return labels

In [ ]:
# 3. Execution (K=3 because there are 3 wine varieties)
kmeans = KMeansCustom(K=3)
cluster_assigned = kmeans.fit(X_scaled)

In [ ]:
# 4. Evaluation (Comparing clusters to actual varieties)
# Since cluster 0 might be variety 2, we use a 'completeness score' 
# to see if the groups matched the labels.
score = completeness_score(y, cluster_assigned)

print(f"\nK-Means Completeness Score: {score:.4f}")
print("(A score closer to 1.0 means the clusters match the varieties perfectly)")

In [ ]:
# 5. Visualization
# We'll plot using two features (Alcohol and Color Intensity) to see the clusters
plt.figure(figsize=(10, 6))
plt.scatter(X_scaled[:, 0], X_scaled[:, 9], c=cluster_assigned, cmap='viridis', alpha=0.6)
plt.scatter(kmeans.centroids[:, 0], kmeans.centroids[:, 9], s=200, c='red', marker='X', label='Centroids')
plt.title("K-Means Clustering: Features 0 (Alcohol) vs 9 (Color Intensity)")
plt.xlabel("Alcohol (Scaled)")
plt.ylabel("Color Intensity (Scaled)")
plt.legend()
plt.grid(True, alpha=0.3)
plt.show()

In [ ]:
Converged at iteration 9

K-Means Completeness Score: 0.8730
(A score closer to 1.0 means the clusters match the varieties perfectly)

![K-Means](KM.png)